# 🎬 Sentiment Analysis using LSTM Deep Learning

This notebook demonstrates how to build a sentiment analysis model using LSTM (Long Short-Term Memory) neural networks on the IMDB movie reviews dataset.

## Table of Contents
1. [Import Libraries](#1-import-libraries)
2. [Load Dataset](#2-load-dataset)
3. [Data Exploration](#3-data-exploration)
4. [Data Preprocessing](#4-data-preprocessing)
5. [Tokenization and Padding](#5-tokenization-and-padding)
6. [Build LSTM Model](#6-build-lstm-model)
7. [Train Model](#7-train-model)
8. [Evaluate Model](#8-evaluate-model)
9. [Save Model and Tokenizer](#9-save-model-and-tokenizer)
10. [Test Predictions](#10-test-predictions)

## 1. Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import re

# Sklearn for preprocessing and metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# TensorFlow/Keras for deep learning
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")

## 2. Load Dataset

We'll use the IMDB Movie Reviews dataset which contains 50,000 movie reviews labeled as positive or negative.

In [ ]:
# Option 1: Load from CSV (if you have the dataset locally)
# df = pd.read_csv('IMDB Dataset.csv')

# Option 2: Load directly from Keras datasets
from tensorflow.keras.datasets import imdb

# Load the IMDB dataset
vocab_size = 10000
(X_train_raw, y_train), (X_test_raw, y_test) = imdb.load_data(num_words=vocab_size)

print(f"Training samples: {len(X_train_raw)}")
print(f"Testing samples: {len(X_test_raw)}")

### Alternative: Load from CSV file

In [ ]:
# If using CSV dataset
# Uncomment this section if you're loading from a CSV file

# df = pd.read_csv('IMDB Dataset.csv')
# print(f"Dataset Shape: {df.shape}")
# df.head()

## 3. Data Exploration

In [ ]:
# Check class distribution
print("Training set class distribution:")
print(f"Positive reviews: {np.sum(y_train == 1)}")
print(f"Negative reviews: {np.sum(y_train == 0)}")

# Visualize class distribution
plt.figure(figsize=(8, 5))
labels = ['Negative', 'Positive']
train_counts = [np.sum(y_train == 0), np.sum(y_train == 1)]
plt.bar(labels, train_counts, color=['#ff6b6b', '#4ecdc4'])
plt.title('Distribution of Sentiment Classes in Training Data')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()

In [ ]:
# Analyze review lengths
train_lengths = [len(review) for review in X_train_raw]

plt.figure(figsize=(10, 5))
plt.hist(train_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.title('Distribution of Review Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.axvline(x=200, color='red', linestyle='--', label='Max Length (200)')
plt.legend()
plt.show()

print(f"Mean review length: {np.mean(train_lengths):.2f}")
print(f"Max review length: {np.max(train_lengths)}")
print(f"Min review length: {np.min(train_lengths)}")

## 4. Data Preprocessing

For the Keras IMDB dataset, the data is already preprocessed. If using your own dataset, you would need to clean the text.

In [ ]:
# Text cleaning function (for custom datasets)
def clean_text(text):
    """
    Clean and preprocess text data.
    
    Args:
        text (str): Raw text input
    
    Returns:
        str: Cleaned text
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Example usage (if using custom dataset)
# df['cleaned_review'] = df['review'].apply(clean_text)

## 5. Tokenization and Padding

### Why Tokenization?
Neural networks work with numbers, not text. Tokenization converts text into sequences of integers.

### Why Padding?
Neural networks require fixed-size inputs. Padding ensures all sequences have the same length.

In [ ]:
# Constants
MAX_SEQUENCE_LENGTH = 200
VOCAB_SIZE = 10000

In [ ]:
# For Keras IMDB dataset, data is already tokenized
# We need to pad the sequences

X_train = pad_sequences(X_train_raw, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_test = pad_sequences(X_test_raw, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
# For custom datasets, you would create and fit a tokenizer
# This is important for deployment as we need to save the tokenizer

# Create and fit tokenizer (for custom text data)
# tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
# tokenizer.fit_on_texts(text_data)

# For demonstration, let's create a tokenizer that can decode IMDB data
word_index = imdb.get_word_index()

# Create reverse mapping
reverse_word_index = {v: k for k, v in word_index.items()}

# Decode a sample review
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

print("Sample decoded review:")
print(decode_review(X_train_raw[0])[:500] + "...")

In [ ]:
# Create a tokenizer for use in deployment
# This tokenizer will be saved and used in the Streamlit app

# Build word index with offset (IMDB uses offset of 3)
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')

# Create word_index from IMDB word index
adjusted_word_index = {k: v + 3 for k, v in word_index.items()}
adjusted_word_index['<PAD>'] = 0
adjusted_word_index['<START>'] = 1
adjusted_word_index['<OOV>'] = 2

tokenizer.word_index = adjusted_word_index

print(f"Vocabulary size: {len(tokenizer.word_index)}")

## 6. Build LSTM Model

### Model Architecture:
- **Embedding Layer**: Converts integer sequences to dense vectors
- **LSTM Layer**: Captures sequential dependencies in the text
- **Dense Layers**: Final classification layers
- **Output Layer**: Sigmoid activation for binary classification

In [ ]:
# Model hyperparameters
EMBEDDING_DIM = 128
LSTM_UNITS = 128
DENSE_UNITS = 64
DROPOUT_RATE = 0.5

In [ ]:
# Build the LSTM model
model = Sequential([
    # Embedding layer: converts integers to dense vectors
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH
    ),
    
    # LSTM layer: captures sequential dependencies
    LSTM(units=LSTM_UNITS, return_sequences=False),
    
    # Dropout layer: prevents overfitting
    Dropout(DROPOUT_RATE),
    
    # Dense layer: fully connected layer
    Dense(units=DENSE_UNITS, activation='relu'),
    
    # Dropout layer
    Dropout(DROPOUT_RATE),
    
    # Output layer: sigmoid for binary classification
    Dense(units=1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Model summary
model.summary()

In [ ]:
# Visualize model architecture
tf.keras.utils.plot_model(
    model,
    to_file='model_architecture.png',
    show_shapes=True,
    show_layer_names=True
)

## 7. Train Model

In [ ]:
# Training hyperparameters
BATCH_SIZE = 64
EPOCHS = 10
VALIDATION_SPLIT = 0.2

In [ ]:
# Define callbacks
callbacks = [
    # Early stopping to prevent overfitting
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),
    # Save best model
    ModelCheckpoint(
        filepath='best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    )
]

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## 8. Evaluate Model

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

In [ ]:
# Generate predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

In [ ]:
# Classification Report
print("Classification Report:")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 9. Save Model and Tokenizer

Save the trained model and tokenizer for deployment in the Streamlit application.

In [ ]:
# Save the model
model.save('sentiment_model.h5')
print("Model saved to 'sentiment_model.h5'")

# Save the tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Tokenizer saved to 'tokenizer.pkl'")

In [ ]:
# Verify saved files
import os

print("Saved files:")
for file in ['sentiment_model.h5', 'tokenizer.pkl']:
    if os.path.exists(file):
        size = os.path.getsize(file) / (1024 * 1024)  # Convert to MB
        print(f"  - {file}: {size:.2f} MB")

## 10. Test Predictions

Test the model with some sample reviews to verify it works correctly.

In [ ]:
# Load saved model and tokenizer for testing
from tensorflow.keras.models import load_model

loaded_model = load_model('sentiment_model.h5')

with open('tokenizer.pkl', 'rb') as f:
    loaded_tokenizer = pickle.load(f)

In [ ]:
def predict_sentiment(text, model, tokenizer, max_length=200):
    """
    Predict sentiment for a given text.
    
    Args:
        text (str): Input text
        model: Trained LSTM model
        tokenizer: Fitted tokenizer
        max_length (int): Maximum sequence length
    
    Returns:
        tuple: (sentiment_label, confidence_score)
    """
    # Tokenize and pad
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')
    
    # Predict
    prediction = model.predict(padded, verbose=0)[0][0]
    
    # Determine sentiment
    if prediction > 0.5:
        return "Positive 😊", prediction
    else:
        return "Negative 😞", 1 - prediction

In [ ]:
# Test with sample reviews
test_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the plot was engaging.",
    "Terrible film. Complete waste of time. The story made no sense and acting was awful.",
    "An average movie with some good moments but overall nothing special.",
    "I loved every minute of this masterpiece. A must-watch for everyone!",
    "Boring, predictable, and way too long. I almost fell asleep."
]

print("Sample Predictions:")
print("=" * 80)

for review in test_reviews:
    sentiment, confidence = predict_sentiment(review, loaded_model, loaded_tokenizer)
    print(f"\nReview: {review[:70]}...")
    print(f"Sentiment: {sentiment} (Confidence: {confidence:.2%})")
    print("-" * 80)

## 📝 Summary

In this notebook, we:

1. **Loaded** the IMDB 50K movie reviews dataset
2. **Explored** the data distribution and characteristics
3. **Preprocessed** the text data (tokenization, padding)
4. **Built** an LSTM deep learning model
5. **Trained** the model with early stopping
6. **Evaluated** model performance
7. **Saved** the model and tokenizer for deployment
8. **Tested** predictions on sample reviews

### Next Steps:
- Move `sentiment_model.h5` and `tokenizer.pkl` to the `model/` directory
- Run the Streamlit app using `streamlit run app.py`
- Access the web interface at `http://localhost:8501`

---
**Created for Academic Machine Learning Project**